# SAFFIRE-V fire-spread relationships

This analysis uses SAFFIRE-V rows from `astroflame_formatted_cgs.csv`. It uses the recorded sample dimensions, not BASS's assumed 10 cm sample length: SAFFIRE-V planform area is calculated as `sample_width_cm * sample_length_cm`, with thickness reported separately. FSR is in cm/s.

The source contains 18 approximate, visually digitized FSR observations from one SAFFIRE-V experiment, at two airflow settings. Thickness and initial oxygen are constant in these observations, so their independent regression effects cannot be estimated. The notebook compares airflow-only, rib-width, rib-width interaction, and quadratic-rib-width linear regressions using leave-one-observation-out error as a within-burn interpolation check. This is not independent experiment-level validation because there is only one experiment group.

## Using BASS results as reference values

BASS can provide context or a separately labeled baseline, but directly filling missing SAFFIRE values with BASS predictions risks domain shift: BASS is flat PMMA at much smaller dimensions, while SAFFIRE-V is a large structured/ribbed PMMA sample. BASS's thickness and oxygen ranges do not cover the SAFFIRE-V values, BASS has no rib-width predictor, and the datasets differ in geometry, pressure reporting, and measurement method. BASS coefficients or predictions therefore should not be presented as measured SAFFIRE values or used as if validated for SAFFIRE. Any transfer should be labeled as an assumption, tested separately, and accompanied by sensitivity analysis.

The digitized SAFFIRE values are approximate and the observations share one burn/test context, so rows are not independent replicates. The regression is descriptive and exploratory, not causal.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

input_name = "astroflame_formatted_cgs.csv"
candidate_dirs = [Path.cwd(), Path.cwd() / "elig-data"]
input_path = next(
    (directory / input_name for directory in candidate_dirs if (directory / input_name).is_file()),
    None,
)
if input_path is None:
    raise FileNotFoundError(f"Could not find {input_name} in the current folder or its elig-data subfolder.")

data_dir = input_path.parent
all_data = pd.read_csv(input_path)
required_columns = {
    "dataset",
    "experiment_id",
    "observation_id",
    "thickness_cm",
    "sample_width_cm",
    "sample_length_cm",
    "rib_width_cm",
    "fsr_cm_s",
    "airflow_cm_s",
    "oxygen_initial_pct",
}
missing_columns = required_columns.difference(all_data.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

dataset_names = all_data["dataset"].astype("string").str.strip().str.upper()
saffire = all_data.loc[dataset_names.isin({"SAFFIRE-V"})].copy()
bass_reference = all_data.loc[dataset_names.str.startswith("BASS", na=False)].copy()
if saffire.empty:
    raise ValueError("No SAFFIRE-V observations were found in the formatted CSV.")
if bass_reference.empty:
    raise ValueError("BASS rows are required for the requested contextual range comparison.")

numeric_columns = [
    "thickness_cm",
    "sample_width_cm",
    "sample_length_cm",
    "rib_width_cm",
    "fsr_cm_s",
    "airflow_cm_s",
    "oxygen_initial_pct",
]
for column in numeric_columns:
    saffire[column] = pd.to_numeric(saffire[column], errors="raise")
if saffire[numeric_columns].isna().any().any():
    missing = saffire[numeric_columns].columns[saffire[numeric_columns].isna().any()].tolist()
    raise ValueError(f"SAFFIRE-V has missing model or dimension values in: {missing}")
if not np.isfinite(saffire[numeric_columns].to_numpy(dtype=float)).all():
    raise ValueError("SAFFIRE-V contains non-finite model or dimension values.")
if (saffire[["thickness_cm", "sample_width_cm", "sample_length_cm", "rib_width_cm"]] <= 0).any().any():
    raise ValueError("SAFFIRE-V sample dimensions must be positive.")

saffire["planform_area_cm2"] = saffire["sample_width_cm"] * saffire["sample_length_cm"]
saffire["sample_volume_cm3"] = saffire["planform_area_cm2"] * saffire["thickness_cm"]

# Keep the original BASS notebook's 10 cm length assumption only for contextual comparison.
bass_reference["sample_width_cm"] = pd.to_numeric(bass_reference["sample_width_cm"], errors="raise")
bass_reference["bass_comparison_area_cm2"] = bass_reference["sample_width_cm"] * 10.0

print(f"SAFFIRE-V observations: {len(saffire)}")
print(f"SAFFIRE-V experiment IDs: {saffire['experiment_id'].nunique()}")
print("SAFFIRE-V sample dimensions (cm):")
print(
    saffire[["sample_width_cm", "sample_length_cm", "thickness_cm"]]
    .drop_duplicates()
    .to_string(index=False)
)
print(f"Planform area: {saffire['planform_area_cm2'].iloc[0]:.1f} cm^2")
print(f"Rectangular envelope volume: {saffire['sample_volume_cm3'].iloc[0]:.1f} cm^3")
print("The area/volume are the outer sample envelope; they do not model rib surface area.")

range_rows = []
bass_reference["thickness_cm"] = pd.to_numeric(bass_reference["thickness_cm"], errors="raise")
bass_reference["oxygen_initial_pct"] = pd.to_numeric(bass_reference["oxygen_initial_pct"], errors="raise")
bass_reference["airflow_cm_s"] = pd.to_numeric(bass_reference["airflow_cm_s"], errors="raise")
for label, saffire_column, bass_column in [
    ("Thickness (cm)", "thickness_cm", "thickness_cm"),
    ("Initial oxygen (%)", "oxygen_initial_pct", "oxygen_initial_pct"),
    ("Airflow (cm/s)", "airflow_cm_s", "airflow_cm_s"),
    ("Planform area (cm^2)", "planform_area_cm2", "bass_comparison_area_cm2"),
]:
    saffire_values = saffire[saffire_column].dropna()
    bass_values = bass_reference[bass_column].dropna()
    range_rows.append(
        {
            "measure": label,
            "SAFFIRE-V min": saffire_values.min(),
            "SAFFIRE-V max": saffire_values.max(),
            "BASS reference min": bass_values.min(),
            "BASS reference max": bass_values.max(),
        }
    )
range_comparison = pd.DataFrame(range_rows)
print("\nSaffire/BASS ranges (BASS area uses the prior notebook's 10 cm length assumption):")
print(range_comparison.to_string(index=False, float_format=lambda value: f"{value:.4g}"))
print("\nUnique Saffire airflow settings:", sorted(saffire["airflow_cm_s"].unique()))
print("Unique Saffire rib widths (cm):", sorted(saffire["rib_width_cm"].unique()))


In [ ]:
saffire["airflow_x_rib_width"] = saffire["airflow_cm_s"] * saffire["rib_width_cm"]
saffire["rib_width_squared"] = saffire["rib_width_cm"] ** 2

requested_features = [
    "airflow_cm_s",
    "thickness_cm",
    "oxygen_initial_pct",
    "rib_width_cm",
]
constant_features = [
    feature for feature in requested_features if saffire[feature].nunique(dropna=True) < 2
]
requested_design = np.column_stack(
    [np.ones(len(saffire))]
    + [saffire[feature].to_numpy(dtype=float) for feature in requested_features]
)
requested_rank = int(np.linalg.matrix_rank(requested_design))
print(
    f"Full requested design (airflow, thickness, oxygen, rib width): "
    f"rank {requested_rank} of {requested_design.shape[1]} columns."
)
print("Constant features that cannot receive independent coefficients:", constant_features)
print("These fixed inputs are documented but omitted from fitted coefficients.")

def build_design(data, features):
    return np.column_stack(
        [np.ones(len(data))]
        + [data[feature].to_numpy(dtype=float) for feature in features]
    )

def leave_one_observation_out(features):
    target = saffire["fsr_cm_s"].to_numpy(dtype=float)
    predictions = np.empty(len(saffire), dtype=float)
    for held_out in range(len(saffire)):
        keep = np.arange(len(saffire)) != held_out
        training = saffire.iloc[keep]
        validation = saffire.iloc[[held_out]]
        training_design = build_design(training, features)
        if np.linalg.matrix_rank(training_design) != training_design.shape[1]:
            raise ValueError(f"Rank-deficient leave-one-out fold for features {features}.")
        coefficients = np.linalg.lstsq(training_design, target[keep], rcond=None)[0]
        predictions[held_out] = (build_design(validation, features) @ coefficients)[0]
    return predictions

def fit_linear_model(features):
    design = build_design(saffire, features)
    rank = int(np.linalg.matrix_rank(design))
    if rank != design.shape[1]:
        raise ValueError(f"Rank-deficient design for features {features}: rank {rank}/{design.shape[1]}.")
    target = saffire["fsr_cm_s"].to_numpy(dtype=float)
    coefficients = np.linalg.lstsq(design, target, rcond=None)[0]
    predicted = design @ coefficients
    residuals = target - predicted
    total_sum_squares = np.sum((target - target.mean()) ** 2)
    r_squared = 1 - np.sum(residuals**2) / total_sum_squares if total_sum_squares else np.nan
    equation = f"FSR = {coefficients[0]:.6g}"
    for coefficient, feature in zip(coefficients[1:], features):
        equation += f" + ({coefficient:.6g} * {feature})"
    loo_predictions = leave_one_observation_out(features)
    return {
        "features": features,
        "coefficients": coefficients,
        "predicted": predicted,
        "loo_predictions": loo_predictions,
        "rank": rank,
        "rmse_cm_s": float(np.sqrt(np.mean(residuals**2))),
        "loo_rmse_cm_s": float(np.sqrt(np.mean((target - loo_predictions) ** 2))),
        "r_squared": float(r_squared),
        "equation": equation,
    }

model_specs = [
    ("Airflow only", ["airflow_cm_s"]),
    ("Airflow plus rib width", ["airflow_cm_s", "rib_width_cm"]),
    ("Airflow plus rib width and interaction", ["airflow_cm_s", "rib_width_cm", "airflow_x_rib_width"]),
    ("Airflow plus linear and quadratic rib width", ["airflow_cm_s", "rib_width_cm", "rib_width_squared"]),
]
model_results = []
for model_name, features in model_specs:
    result = fit_linear_model(features)
    result["model"] = model_name
    model_results.append(result)

best_saffire_model = min(model_results, key=lambda result: result["loo_rmse_cm_s"])

model_summary = pd.DataFrame(
    [
        {
            "model": result["model"],
            "predictors": ", ".join(result["features"]),
            "rank": result["rank"],
            "training_R2": result["r_squared"],
            "training_RMSE_cm_s": result["rmse_cm_s"],
            "LOO_RMSE_cm_s": result["loo_rmse_cm_s"],
            "equation": result["equation"],
        }
        for result in model_results
    ]
)
print("\nExploratory linear-model comparison (row-wise LOO is within-burn only):")
print(model_summary.to_string(index=False, float_format=lambda value: f"{value:.6g}"))
print(f"\nLowest row-wise LOO RMSE: {best_saffire_model['model']} ({best_saffire_model['loo_rmse_cm_s']:.6g} cm/s).")
print(
    "\nAirflow/rib-width Pearson correlation:",
    f"{saffire[['airflow_cm_s', 'rib_width_cm']].corr().iloc[0, 1]:.4f}",
)
print(
    "No independent validation score is reported: the observations represent "
    f"{saffire['experiment_id'].nunique()} experiment ID(s), not independent test replicates."
)


## Interpretation and transfer cautions

- The large sample footprint is explicitly computed from the recorded 40 cm by 20 cm dimensions. It is not substituted with BASS's assumed 10 cm length. FSR is already a spread speed; do not multiply or divide it by sample area without a separately justified physical model.
- Thickness (1 cm) and initial oxygen (26.9%) do not vary in this Saffire subset. Their effects are inseparable from the intercept, so this notebook does not claim coefficients for them. The full requested design matrix is rank-deficient.
- The added rib-width regression is an exploratory within-dataset association. A quadratic rib-width term has the lowest row-wise leave-one-out RMSE among the tested candidates, but held-out rows come from the same burn; this is only an interpolation diagnostic, not independent validation. The fit can be sensitive to individual rows and cannot establish causation or general performance.
- BASS has no rib-width field and its recorded ranges differ in thickness and oxygen. A BASS-based imputation would require unverified transfer assumptions; label such values as borrowed predictions, never as observed Saffire measurements, and report a sensitivity range rather than a single certain value.

In [ ]:
import matplotlib.pyplot as plt

fig, axis = plt.subplots(figsize=(9, 6))
for airflow, group in saffire.groupby("airflow_cm_s"):
    group = group.sort_values("rib_width_cm")
    axis.plot(
        group["rib_width_cm"],
        group["fsr_cm_s"],
        marker="o",
        linewidth=1.2,
        label=f"Airflow {airflow:g} cm/s",
    )
axis.set(
    title="SAFFIRE-V observed FSR by rib width",
    xlabel="Rib width (cm)",
    ylabel="Digitized FSR (cm/s)",
)
axis.legend()
axis.grid(alpha=0.25)
fig.tight_layout()
rib_chart = data_dir / "saffire_fsr_by_rib_width.png"
fig.savefig(rib_chart, dpi=180, bbox_inches="tight")
plt.show()

fig, axis = plt.subplots(figsize=(7, 7))
for result in model_results:
    axis.scatter(
        saffire["fsr_cm_s"],
        result["loo_predictions"],
        alpha=0.8,
        label=result["model"],
    )
limits = [float(saffire["fsr_cm_s"].min()), float(saffire["fsr_cm_s"].max())]
axis.plot(limits, limits, color="black", linestyle="--", label="Perfect fit")
axis.set(
    title="SAFFIRE-V leave-one-observation-out fits (same burn)",
    xlabel="Observed digitized FSR (cm/s)",
    ylabel="Fitted FSR (cm/s)",
)
axis.legend()
axis.grid(alpha=0.25)
fig.tight_layout()
regression_chart = data_dir / "saffire_fsr_regression_comparison.png"
fig.savefig(regression_chart, dpi=180, bbox_inches="tight")
plt.show()

report_lines = [
    "SAFFIRE-V exploratory fire-spread analysis",
    f"Observations: {len(saffire)}; experiment IDs: {saffire['experiment_id'].nunique()}",
    f"Sample dimensions (cm): {saffire['sample_width_cm'].iloc[0]:g} x {saffire['sample_length_cm'].iloc[0]:g} x {saffire['thickness_cm'].iloc[0]:g}",
    f"Planform area: {saffire['planform_area_cm2'].iloc[0]:g} cm^2",
    f"Envelope volume: {saffire['sample_volume_cm3'].iloc[0]:g} cm^3",
    f"Fixed predictors: {', '.join(constant_features) if constant_features else 'none'}",
    f"Lowest within-burn leave-one-out RMSE model: {best_saffire_model['model']} ({best_saffire_model['loo_rmse_cm_s']:.6g} cm/s).",
    "Row-wise LOO is not independent experiment-level validation; only one Saffire experiment is available.",
    "BASS values are contextual only; transferring BASS coefficients/predictions to Saffire is unvalidated.",
    "",
    model_summary.to_string(index=False),
]
report_path = data_dir / "saffire_fsr_model_results.txt"
report_path.write_text("\n".join(report_lines) + "\n", encoding="utf-8")
print("Saved Saffire outputs:")
print(rib_chart.name)
print(regression_chart.name)
print(report_path.name)
